[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/01_finetuning_and_architecture/01_memory_budget_verifier.ipynb)

# Armory Notebook 1 — The 16GB Memory Budget, Verified Against Real Hardware

| | |
|---|---|
| **Companion to** | Lesson 1 — Defending QLoRA on ESCI (Refresher 3) |
| **Runtime** | Colab **T4 GPU** (~15 GB) for the live section; the arithmetic section runs anywhere |
| **Estimated time** | 20 minutes |
| **XP** | **0.** Armory notebooks produce the evidence you defend in drills — the defense is what pays. |
| **Last verified** | 2026-07-13 |

The interviewer's memory question ("walk me through your 16GB — byte counts") is checkable arithmetic. This notebook makes you do the arithmetic *first*, then loads Qwen2.5-1.5B in NF4 and confronts your prediction with `torch.cuda` allocator numbers. The gap between predicted and observed is where the follow-up questions live.

> [!IMPORTANT]
> Numbers this notebook prints on a Colab T4 describe *this notebook's* inference-time load — **not** your training run's peak. Your training peak includes activations, gradients, and optimizer states for the adapters at your batch/sequence config. Record vault entries only from your own training logs.

## Part 1 — The paper budget (runs on any machine, no GPU)

Predict before you measure. The function below reproduces Lesson 1's Refresher 3 as executable arithmetic.

In [ ]:
GIB = 1024**3

def full_finetune_budget(n_params: float) -> dict:
    """Mixed-precision AdamW: bf16 weights/grads, fp32 master weights + two fp32 moments."""
    return {
        "bf16 weights":        2 * n_params / GIB,
        "bf16 gradients":      2 * n_params / GIB,
        "fp32 master weights": 4 * n_params / GIB,
        "fp32 Adam moment m":  4 * n_params / GIB,
        "fp32 Adam moment v":  4 * n_params / GIB,
    }

def qlora_budget(n_params: float, adapter_params: float) -> dict:
    """NF4 base (0.5 byte/param) + double-quant constants (~0.127 bit/param)
    + bf16 adapters carrying bf16 grads and fp32 AdamW states."""
    return {
        "NF4 base weights":       0.5 * n_params / GIB,
        "quantization constants": (0.127 / 8) * n_params / GIB,
        "bf16 adapter weights":   2 * adapter_params / GIB,
        "bf16 adapter grads":     2 * adapter_params / GIB,
        "fp32 adapter optimizer": 12 * adapter_params / GIB,  # master copy + m + v
    }

def show(title: str, budget: dict) -> float:
    print(f"\n{title}")
    total = 0.0
    for line_item, gib in budget.items():
        print(f"  {line_item:<24} {gib:7.2f} GiB")
        total += gib
    print(f"  {'TOTAL (before activations)':<24} {total:7.2f} GiB")
    return total

QWEN_PARAMS = 1.54e9        # Qwen2.5-1.5B
ADAPTER_PARAMS = 30e6       # placeholder — replace with YOUR adapter count below

ft_total = show("FULL FINE-TUNING, Qwen2.5-1.5B", full_finetune_budget(QWEN_PARAMS))
ql_total = show("QLoRA, same model", qlora_budget(QWEN_PARAMS, ADAPTER_PARAMS))
print(f"\nVerdict: full FT needs {ft_total:.1f} GiB of state on a 16 GiB card -> impossible.")
print(f"QLoRA needs {ql_total:.1f} GiB of state -> activations become the binding constraint.")

**Drill:** before running the next section, write down your predicted allocator reading for loading the NF4 model alone. You are allowed to be wrong; you are not allowed to not have a prediction — that is the difference between deriving and reciting.

## Part 2 — Load the real model in NF4 (GPU required)

Dependency versions below were verified on 2026-07-13; if an install fails months later, check the course CHANGELOG for updated pins.

In [ ]:
%pip install -q "transformers>=4.51,<5.0" "bitsandbytes>=0.45" "accelerate>=1.2" "peft>=0.14"

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA device found. In Colab: Runtime > Change runtime type > T4 GPU. "
        "Part 1's arithmetic still stands - only the live verification needs a GPU."
    )

from transformers import AutoModelForCausalLM, BitsAndBytesConfig

nf4_recipe = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,      # the ~0.127 bit/param line item
    bnb_4bit_compute_dtype=torch.bfloat16,
)

torch.cuda.reset_peak_memory_stats()
base_lm = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B",
    quantization_config=nf4_recipe,
    device_map="auto",
)

loaded_gib = torch.cuda.memory_allocated() / GIB
print(f"Allocator after NF4 load : {loaded_gib:.2f} GiB")
print(f"Paper prediction (base+constants): {0.5*QWEN_PARAMS/GIB + (0.127/8)*QWEN_PARAMS/GIB:.2f} GiB")
print("\nGap analysis is the interview answer: embeddings/lm_head kept in higher precision,")
print("buffer tensors, and allocator granularity all sit between paper and silicon.")

In [ ]:
from peft import LoraConfig, get_peft_model

# Reproduce YOUR configuration here - rank, alpha, and target modules from your run config.
adapter_recipe = LoraConfig(
    r=16,                       # [FILL: your rank]
    lora_alpha=32,              # [FILL: your alpha]
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # [FILL: your targets]
    task_type="CAUSAL_LM",
)

adapted_lm = get_peft_model(base_lm, adapter_recipe)
trainable, total = 0, 0
for p in adapted_lm.parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()

print(f"Trainable params : {trainable:,} ({100*trainable/total:.3f}% of {total:,})")
print("Your resume says '<2%'. This cell is where that claim becomes a number you own.")
print("Re-run Part 1 with ADAPTER_PARAMS set to the trainable count above.")

## What goes in the Metric Vault

| Vault row | Source |
|---|---|
| QLoRA: trainable-param % and count | The cell above, **after** you set your real rank/alpha/targets |
| QLoRA: LoRA rank, alpha, target modules | Your run config (this notebook just re-states it) |
| QLoRA: peak GPU memory observed | **Your training logs only** — this notebook never saw your activations |

If your training logs never captured peak memory, the vault row stays `QUALITATIVE-ONLY` and your interview answer is: "I can derive the budget — here it is — but I didn't log the observed peak; that's instrumentation I'd add." That answer survives. A guessed number doesn't.